## 持久化 

#### checkpointer: 检查点保存器，如内存，数据库
#### checkpoint: 检查点，图的每一个步骤都会生成

START->node_a->node_b->END
每个节点均为一个检查点

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add


class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]


def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}


def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
## 执行图
graph.invoke({"foo": "", "bar": []}, config)

获取最新状态
StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65ba-8002-ce21c67fcf12'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-28T07:13:40.927224+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65b9-8001-528a7db37d64'}}, tasks=(), interrupts=())
---
StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65ba-8002-ce21c67fcf12'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-28T07:13:40.927224+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65b9-8001-528a7db37d64'}}, tasks=(), interrupts=())
---
StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread

In [7]:
##获取指定配置的状态,每个checkpoint有自己的id，配置中可添加checkpointId获取指定线程和checkpoint的状态
s = graph.get_state(config)
print("获取最新状态")
print(s)

## 获取所有的历史状态,从最新的状态开始
his_state = graph.get_state_history(config)
for sta in his_state:
    print("---")
    print(sta)

获取最新状态
StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65ba-8002-ce21c67fcf12'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-28T07:13:40.927224+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65b9-8001-528a7db37d64'}}, tasks=(), interrupts=())
---
StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65ba-8002-ce21c67fcf12'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-28T07:13:40.927224+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc18d-f996-65b9-8001-528a7db37d64'}}, tasks=(), interrupts=())
---
StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread

## 通过checkpointId+线程id执行，会从这个checkpointId对应的节点重新执行，对于之前的节点不重新执行，但是会知道当时的数据，以当时的数据继续执行

In [10]:
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1f0fc18d-f996-65b9-8001-528a7db37d64"}}
graph.invoke(None, config)

{'foo': 'b', 'bar': ['a', 'b']}

### 手动更新状态

In [14]:
config = {"configurable": {"thread_id": "1"}}
## config,需要更新的状态(bar的状态定义了累加器，所以结果进行的是累加),as_node：对这次更新相当于执行了这个节点
graph.update_state(config, {'foo': 'b', 'bar': ['a']}, "node_b")
print(graph.get_state(config))

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b', 'a', 'a', 'a']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc1af-e04c-6e38-8005-3a74afbba477'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2026-01-28T07:28:50.956242+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fc1a8-bc77-62c3-8004-bb471fead13a'}}, tasks=(), interrupts=())


# store 跨线程的存储

对于checkpointer，只是存储当前这个图的状态，而有一些东西是共享的，比如一个用户可能多次调用某个图，那么会对应有多个checkpointer，而store则是总的一个存储

**store的存储和查询也可以使用embed的形式，而不是采用精准匹配**

In [23]:
from langgraph.graph import MessagesState
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore


def put_user_info(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    ##store的命名空间,为元组，命名空间的对象是一个键值对，这里也只是将userid作为一个命名空间的变量
    namespace = (user_id, "anything1", "anything2")
    #在该命名空间放入信息
    store.put(namespace, "user_name", "lkm")
    store.put(namespace, "user_nick_name", "一碰就碎")
    return {"foo": "a", "bar": ["a"]}


def get_user_info(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    namespace = (user_id, "anything1", "anything2")
    memories = store.search(namespace)
    print(memories)


in_memory_store = InMemoryStore()
workflow_store = StateGraph(State)
workflow_store.add_node(put_user_info)
workflow_store.add_node(get_user_info)
workflow_store.add_edge(START, "put_user_info")
workflow_store.add_edge("put_user_info", "get_user_info")
workflow_store.add_edge("get_user_info", END)
graph_with_store = workflow_store.compile(checkpointer=checkpointer, store=in_memory_store)
config = {"configurable": {"thread_id": "2", "user_id": "1"}}

graph_with_store.invoke({}, config)

[Item(namespace=['1', 'anything1', 'anything2'], key='user_name', value='lkm', created_at='2026-01-28T09:12:55.287457+00:00', updated_at='2026-01-28T09:12:55.287457+00:00', score=None), Item(namespace=['1', 'anything1', 'anything2'], key='user_nick_name', value='一碰就碎', created_at='2026-01-28T09:12:55.287457+00:00', updated_at='2026-01-28T09:12:55.287457+00:00', score=None)]


{'foo': 'a', 'bar': ['a']}